**Training**

In [ ]:
!pip install m2cgen
# 1. First, check your PyTorch and CUDA versions (Mamba is very strictly tied to these)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

# 2. Install the build dependencies first
!pip install packaging wheel ninja

# 3. Attempt installation bypassing the isolated build environment
!pip install "causal-conv1d>=1.2.0" --no-build-isolation
!pip install mamba-ssm --no-build-isolation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.signal import savgol_filter
from numpy.polynomial import Polynomial
from sklearn.preprocessing import normalize, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ==========================================
# 0. DEFINE AIR-PLS ALGORITHM
# ==========================================
def baseline_als(y, lam, p, niter=10):
    """
    Asymmetric Least Squares Smoothing for baseline correction.
    Used to approximate airPLS logic.
    """
    L = len(y)
    D = sparse.diags([1,-2,1],[0,-1,-2], shape=(L,L-2))
    D = lam * D.dot(D.transpose())
    w = np.ones(L)
    for i in range(niter):
        W = sparse.spdiags(w, 0, L, L)
        Z = W + D
        z = spsolve(Z, w*y)
        w = p * (y > z) + (1-p) * (y < z)
    return z

def apply_airpls(spectra, lam=1e5, p=0.01):
    """Applies ALS baseline correction to a matrix of spectra."""
    corrected = np.zeros_like(spectra)
    for i in range(spectra.shape[0]):
        baseline = baseline_als(spectra[i], lam, p)
        corrected[i] = spectra[i] - baseline
    return np.maximum(corrected, 0) # Ensure non-negative

# ==========================================
# 1. LOAD DATASET
# ==========================================
filepath = "/kaggle/input/datasets/ryancclh/irish-soil-dataset/Irish_Soil_DataSet2.csv"
print("Loading data...")
df = pd.read_csv(filepath, index_col=0)
X_raw = df.T.values.astype(float)
wavelengths = df.T.columns.values.astype(float)
sample_names = df.T.index.values

# Generate Task B Labels (Contamination)
y = np.array([1 if any(c in name for c in ["ChalkT", "Arsenic", "TPH", "GardenTop_w_HeavyMetals"]) else 0 for name in sample_names])

# ==========================================
# 2. DEFINE THE THREE PREPROCESSING STATES
# ==========================================

# STATE 1: Raw Spectra (No Preprocessing)
print("Processing State 1: Raw Spectra...")
X_state1 = X_raw.copy()

# STATE 2: Bad Preprocessing (airPLS + Min-Max)
print("Processing State 2: airPLS + Min-Max...")
X_airpls = apply_airpls(X_raw)
X_state2 = MinMaxScaler().fit_transform(X_airpls)

# STATE 3: Champion Pipeline (Poly-5 + SG + L2)
print("Processing State 3: Champion Pipeline...")
X_base = np.zeros_like(X_raw)
for i in range(X_raw.shape[0]):
    poly = Polynomial.fit(wavelengths, X_raw[i], deg=5)
    X_base[i] = np.maximum(savgol_filter(np.maximum(X_raw[i] - poly(wavelengths), 0), window_length=15, polyorder=3), 0)
X_state3 = normalize(X_base, norm='l2', axis=1)

# ==========================================
# 3. TRAIN & EVALUATE RANDOM FOREST
# ==========================================
states_data = {
    'Raw Spectra\n(No Preprocessing)': X_state1,
    'Sub-optimal Pipeline\n(airPLS + Min-Max)': X_state2,
    'Champion Pipeline\n(Poly-5 + SG + L2)': X_state3
}

accuracies = []
state_names = list(states_data.keys())

print("\nTraining models and computing accuracies...")
for name, X_data in states_data.items():
    X_train, X_test, y_train, y_test = train_test_split(X_data, y, test_size=0.2, random_state=42, stratify=y)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred) * 100
    accuracies.append(acc)

# ==========================================
# 4. GENERATE THE "PROOF OF VALUE" PLOT
# ==========================================
plt.figure(figsize=(9, 6))
colors = ['#d3d3d3', '#ff9999', '#66b3ff']  

bars = plt.bar(state_names, accuracies, color=colors, width=0.6, edgecolor='black')

baseline_acc = 73.01
plt.axhline(y=baseline_acc, color='r', linestyle='--', linewidth=2, label=f'PLS-DA Baseline ({baseline_acc}%)')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1.5, f'{yval:.1f}%', 
             ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.ylim(0, 105)
plt.ylabel('Random Forest Accuracy (%)', fontsize=12, fontweight='bold')
plt.title('Impact of Preprocessing Pipelines on Classification Performance', fontsize=14, fontweight='bold', pad=15)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.legend(loc='upper left', fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('preprocessing_impact_airpls.png', dpi=300)
plt.show()

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import joblib
import shutil
import tensorflow as tf
import m2cgen as m2c
from scipy.signal import savgol_filter
from numpy.polynomial import Polynomial
from sklearn.preprocessing import normalize, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from tensorflow.keras.layers import Input, Dense, Conv1D, LeakyReLU, MaxPooling1D, Flatten, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

import torch
import torch.nn as nn
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
except ImportError:
    MAMBA_AVAILABLE = False

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ==============================================================================
# DIRECTORY SETUP
# ==============================================================================
TASKA_DIR = "FYP_Exported_Models_TaskA"
TASKB_DIR = "FYP_Exported_Models_TaskB"
TASKA_EXT_DIR = "FYP_Exported_Models_TaskA_Ext"

os.makedirs(TASKA_DIR, exist_ok=True)
os.makedirs(TASKB_DIR, exist_ok=True)
os.makedirs(TASKA_EXT_DIR, exist_ok=True)

# Helper function for CNN architecture
def build_inception_cnn(input_shape, num_classes):
    inp = Input(shape=input_shape)
    x = LeakyReLU(0.3)(Conv1D(8, 3, padding='same', kernel_regularizer=l2(0.01))(inp))
    merged = Concatenate()([Flatten()(LeakyReLU(0.3)(Conv1D(4, k, padding='same')(x))) for k in [1, 3, 5]] + [Flatten()(MaxPooling1D(3, strides=1, padding='same')(x))])
    out = Dense(num_classes, activation='softmax')(Dropout(0.1)(Dense(64, activation='relu')(merged)))
    m = Model(inp, out)
    m.compile(optimizer=Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def generate_nutrient_labels(X, w_array, peaks_config):
    aggregated_areas = np.zeros(X.shape[0])
    for peak in peaks_config:
        mask = (w_array >= peak["centre"] - peak["tol"]) & (w_array <= peak["centre"] + peak["tol"])
        if np.any(mask):
            aggregated_areas += (np.sum(X[:, mask], axis=1) * peak["weight"])
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(aggregated_areas.reshape(-1, 1))
    sorted_centers = np.argsort(kmeans.cluster_centers_.flatten())
    label_map = {old: new for new, old in enumerate(sorted_centers)}
    return np.vectorize(label_map.get)(kmeans.labels_)

nutrient_configs = {
    'N': [{"centre": 821.634, "tol": 0.3, "weight": 1.0}, {"centre": 868.340, "tol": 0.3, "weight": 1.0}, {"centre": 868.028, "tol": 0.3, "weight": 1.0}, {"centre": 496.398, "tol": 0.3, "weight": 0.5}, {"centre": 746.831, "tol": 0.3, "weight": 0.2}],
    'P': [{"centre": 213.618, "tol": 0.3, "weight": 1.0}, {"centre": 213.547, "tol": 0.3, "weight": 0.5}, {"centre": 253.561, "tol": 0.3, "weight": 1.0}, {"centre": 255.325, "tol": 0.3, "weight": 0.5}],
    'K': [{"centre": 766.490, "tol": 0.3, "weight": 1.0}, {"centre": 769.896, "tol": 0.3, "weight": 1.0}, {"centre": 404.414, "tol": 0.3, "weight": 0.5}, {"centre": 404.721, "tol": 0.3, "weight": 0.5}]
}

# ==============================================================================
# INTERNAL DATASET PREP (Tasks A & B)
# ==============================================================================
filepath = "/kaggle/input/datasets/ryancclh/irish-soil-dataset/Irish_Soil_DataSet2.csv"
print(f"Loading Internal data from {filepath}...")
df = pd.read_csv(filepath, index_col=0)
X_raw = df.T.values.astype(float)
wavelengths = df.T.columns.values.astype(float)
sample_names = df.T.index.values

print("Applying Preprocessing to Internal Data...")
X_base = np.zeros_like(X_raw)
for i in range(X_raw.shape[0]):
    p = Polynomial.fit(wavelengths, X_raw[i], deg=5)
    X_base[i] = np.maximum(savgol_filter(np.maximum(X_raw[i] - p(wavelengths), 0), window_length=15, polyorder=3), 0)

X_preprocessed = normalize(X_base, norm='l2', axis=1)
X_raw_scaled = MinMaxScaler().fit_transform(X_raw)
X_cnn = normalize(X_raw, norm='l2', axis=1).reshape((X_raw.shape[0], X_raw.shape[1], 1))

np.save(os.path.join(TASKA_DIR, 'X_preprocessed_full.npy'), X_preprocessed)
np.save(os.path.join(TASKA_DIR, 'X_raw_scaled_full.npy'), X_raw_scaled)

# ==============================================================================
# TASK A: INTERNAL TRAINING
# ==============================================================================
print("\n--- TASK A: TRAINING INTERNAL MODELS ---")
pseudo_labels_dict = {nut: generate_nutrient_labels(X_raw, wavelengths, conf) for nut, conf in nutrient_configs.items()}
joblib.dump(pseudo_labels_dict, os.path.join(TASKA_DIR, 'pseudo_labels_dict.joblib'))

inputs = Input(shape=(X_raw_scaled.shape[1],))
latent = Dense(64, activation='relu')(Dense(256, activation='relu')(inputs))
outputs = Dense(X_raw_scaled.shape[1], activation='linear')(Dense(256, activation='relu')(latent))
autoencoder = Model(inputs, outputs)
encoder = Model(inputs, latent)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(X_raw_scaled, X_raw_scaled, epochs=50, batch_size=32, verbose=0)
X_latent = encoder.predict(X_raw_scaled, verbose=0)
encoder.save(os.path.join(TASKA_DIR, "global_encoder.keras"))

for nutrient in ['N', 'P', 'K']:
    print(f"Training Task A (Internal) for: {nutrient}")
    y = pseudo_labels_dict[nutrient]
    
    rf_sel = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_preprocessed, y)
    top_87_idx = np.argsort(rf_sel.feature_importances_)[::-1][:87]
    np.save(os.path.join(TASKA_DIR, f'top87_indices_{nutrient}.npy'), top_87_idx)
    X_phys = X_preprocessed[:, top_87_idx]

    joblib.dump(RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=5, random_state=42, n_jobs=-1).fit(X_phys, y), os.path.join(TASKA_DIR, f"Prep_RF_{nutrient}.joblib"))
    joblib.dump(XGBClassifier(max_depth=6, learning_rate=0.1, random_state=42).fit(X_phys, y), os.path.join(TASKA_DIR, f"Prep_XGB_{nutrient}.joblib"))
    joblib.dump(LGBMClassifier(max_depth=6, learning_rate=0.1, random_state=42, verbose=-1).fit(X_phys, y), os.path.join(TASKA_DIR, f"Prep_LGBM_{nutrient}.joblib"))
    joblib.dump(XGBClassifier(max_depth=6, learning_rate=0.1, random_state=42).fit(X_latent, y), os.path.join(TASKA_DIR, f"AE_XGBoost_{nutrient}.joblib"))


    cnn = build_inception_cnn(input_shape=(X_cnn.shape[1], 1), num_classes=3)
    cnn.fit(X_cnn, y, epochs=1500, batch_size=32, verbose=0) 
    cnn.save(os.path.join(TASKA_DIR, f"Inception_CNN_{nutrient}.keras"))

# 1. Load the model from the file
loaded_model_N = joblib.load(os.path.join(TASKA_DIR, "Prep_LGBM_N.joblib"))

# 2. Pass the model object to m2cgen
c_code_N = m2c.export_to_c(loaded_model_N)

# 3. Write it to a file
with open("Prep_LGBM_N.c", "w") as f:
    f.write(c_code_N)

# 1. Load the model from the file
loaded_model_P = joblib.load(os.path.join(TASKA_DIR, "Prep_LGBM_P.joblib"))

# 2. Pass the model object to m2cgen
c_code_P = m2c.export_to_c(loaded_model_P)

# 3. Write it to a file
with open("Prep_LGBM_P.c", "w") as f:
    f.write(c_code_P)

# 1. Load the model from the file
loaded_model_K = joblib.load(os.path.join(TASKA_DIR, "Prep_LGBM_K.joblib"))

# 2. Pass the model object to m2cgen
c_code_K = m2c.export_to_c(loaded_model_K)

# 3. Write it to a file
with open("Prep_LGBM_K.c", "w") as f:
    f.write(c_code_K)

# ==============================================================================
# TASK B: CONTAMINATION TRAINING
# ==============================================================================
print("\n--- TASK B: TRAINING ---")
y_task_B = np.array([1 if any(c in name for c in ["ChalkT", "Arsenic", "TPH", "GardenTop_w_HeavyMetals"]) else 0 for name in sample_names])

rf_sel_b = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_preprocessed, y_task_B)
top_87_b = np.argsort(rf_sel_b.feature_importances_)[::-1][:87]
np.save(os.path.join(TASKB_DIR, "top87_indices_TaskB.npy"), top_87_b)
X_b_selected = X_preprocessed[:, top_87_b]

joblib.dump(RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1).fit(X_preprocessed, y_task_B), os.path.join(TASKB_DIR, "RandomForest_FullSpectra_TaskB.joblib"))
joblib.dump(RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1).fit(X_b_selected, y_task_B), os.path.join(TASKB_DIR, "RandomForest_GiniTop87_TaskB.joblib"))
joblib.dump(XGBClassifier(max_depth=6, learning_rate=0.1, random_state=42).fit(X_b_selected, y_task_B), os.path.join(TASKB_DIR, "XGBoost_GiniTop87_TaskB.joblib"))
joblib.dump(LGBMClassifier(max_depth=6, learning_rate=0.1, random_state=42, verbose=-1).fit(X_b_selected, y_task_B), os.path.join(TASKB_DIR, "LightGBM_GiniTop87_TaskB.joblib"))
joblib.dump(RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1).fit(X_latent, y_task_B), os.path.join(TASKB_DIR, "RandomForest_AELatent_TaskB.joblib"))

cnn_b = build_inception_cnn(input_shape=(X_cnn.shape[1], 1), num_classes=2)
cnn_b.fit(X_cnn, y_task_B, epochs=1500, batch_size=32, verbose=0)
cnn_b.save(os.path.join(TASKB_DIR, "InceptionCNN_FullSpectra_TaskB.keras"))

if MAMBA_AVAILABLE:
    class MambaClassifier(nn.Module):
        def __init__(self, input_dim, num_cls=2):
            super().__init__()
            self.proj = nn.Linear(input_dim, 64)
            self.mamba = Mamba(d_model=64, d_state=16, d_conv=4, expand=2)
            self.clf = nn.Linear(64, num_cls)
        def forward(self, x):
            return self.clf(self.mamba(self.proj(x).unsqueeze(1)).squeeze(1))

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    m_model = MambaClassifier(X_cnn.shape[1], 2).to(device)
    opt = torch.optim.Adam(m_model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    
    X_tr_t = torch.tensor(X_cnn.squeeze(2), dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_task_B, dtype=torch.long).to(device)
    
    for _ in range(500):
        m_model.train()
        opt.zero_grad()
        crit(m_model(X_tr_t), y_tr_t).backward()
        opt.step()
    torch.save(m_model.state_dict(), os.path.join(TASKB_DIR, "MambaSSM_FullSpectra_TaskB.pth"))


# ==============================================================================
# TASK A: EXTERNAL DATASET TRAINING (P & K)
# ==============================================================================
print("\n--- TASK A: TRAINING EXTERNAL MODELS (P & K) ---")

LIBS_DATA_PATH = "/kaggle/input/datasets/ryancclh/libs-extra-dataset/LIBS Extra Dataset/LIBS data.xlsx"
df_libs = pd.read_excel(LIBS_DATA_PATH, header=None)
wavelengths_ext = df_libs.iloc[2:, 0].values.astype(float)
X_raw_ext = df_libs.iloc[2:, 1:].values.astype(float).T

# Preprocess identically applied to External 
X_smooth_ext = savgol_filter(X_raw_ext, window_length=15, polyorder=3, axis=1)
X_prep_ext = normalize(np.maximum(X_smooth_ext, 0), norm='l2', axis=1)
X_cnn_ext = X_prep_ext.reshape((X_prep_ext.shape[0], X_prep_ext.shape[1], 1))
EXT_FEAT_COUNT = X_prep_ext.shape[1]

# Global Ext Autoencoder
inputs_ext = Input(shape=(EXT_FEAT_COUNT,))
latent_ext = Dense(64, activation='relu')(Dense(256, activation='relu')(inputs_ext))
outputs_ext = Dense(EXT_FEAT_COUNT, activation='linear')(Dense(256, activation='relu')(latent_ext))
ae_ext = Model(inputs_ext, outputs_ext)
encoder_ext = Model(inputs_ext, latent_ext)
ae_ext.compile(optimizer='adam', loss='mse')
ae_ext.fit(X_prep_ext, X_prep_ext, epochs=50, batch_size=32, verbose=0)
X_ext_latent = encoder_ext.predict(X_prep_ext, verbose=0)

encoder_ext.save(os.path.join(TASKA_EXT_DIR, "global_encoder_ext.keras"))

pseudo_ext = {nut: generate_nutrient_labels(X_prep_ext, wavelengths_ext, nutrient_configs[nut]) for nut in ['P', 'K']}
joblib.dump(pseudo_ext, os.path.join(TASKA_EXT_DIR, 'pseudo_labels_ext.joblib'))

for nutrient in ['P', 'K']:
    print(f"Training Task A (External) for: {nutrient} (Features: {EXT_FEAT_COUNT})")
    y_ext = pseudo_ext[nutrient]
    
    rf_sel_ext = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_prep_ext, y_ext)
    top_87_ext = np.argsort(rf_sel_ext.feature_importances_)[::-1][:87]
    np.save(os.path.join(TASKA_EXT_DIR, f'top87_indices_ext_{nutrient}.npy'), top_87_ext)
    
    X_ext_phys = X_prep_ext[:, top_87_ext]

    joblib.dump(RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1).fit(X_ext_phys, y_ext), os.path.join(TASKA_EXT_DIR, f"Prep_RF_ext_{nutrient}.joblib"))
    joblib.dump(XGBClassifier(max_depth=6, learning_rate=0.1, random_state=42).fit(X_ext_phys, y_ext), os.path.join(TASKA_EXT_DIR, f"Prep_XGB_ext_{nutrient}.joblib"))
    joblib.dump(LGBMClassifier(max_depth=6, learning_rate=0.1, random_state=42, verbose=-1).fit(X_ext_phys, y_ext), os.path.join(TASKA_EXT_DIR, f"Prep_LGBM_ext_{nutrient}.joblib"))
    joblib.dump(XGBClassifier(max_depth=6, learning_rate=0.1, random_state=42).fit(X_ext_latent, y_ext), os.path.join(TASKA_EXT_DIR, f"AE_XGBoost_ext_{nutrient}.joblib"))
    
    cnn_ext = build_inception_cnn(input_shape=(EXT_FEAT_COUNT, 1), num_classes=3)
    cnn_ext.fit(X_cnn_ext, y_ext, epochs=1500, batch_size=16, verbose=0)
    cnn_ext.save(os.path.join(TASKA_EXT_DIR, f"Inception_CNN_ext_{nutrient}.keras"))
    
    if MAMBA_AVAILABLE:
        m_model_ext = MambaClassifier(EXT_FEAT_COUNT, 3).to(device)
        opt_ext = torch.optim.Adam(m_model_ext.parameters(), lr=1e-3)
        crit_ext = nn.CrossEntropyLoss()
        
        X_tr_t_ext = torch.tensor(X_prep_ext, dtype=torch.float32).to(device)
        y_tr_t_ext = torch.tensor(y_ext, dtype=torch.long).to(device)
        
        for _ in range(300):
            m_model_ext.train()
            opt_ext.zero_grad()
            crit_ext(m_model_ext(X_tr_t_ext), y_tr_t_ext).backward()
            opt_ext.step()
        torch.save(m_model_ext.state_dict(), os.path.join(TASKA_EXT_DIR, f"MambaSSM_ext_{nutrient}.pth"))

print("\n--- ZIPPING OUTPUTS ---")
shutil.make_archive(TASKA_DIR, 'zip', TASKA_DIR)
shutil.make_archive(TASKB_DIR, 'zip', TASKB_DIR)
shutil.make_archive(TASKA_EXT_DIR, 'zip', TASKA_EXT_DIR)
print("TRAINING & EXPORT PIPELINE COMPLETE.")

In [ ]:
# ==============================================================================
# 5-FOLD OOF ACCUMULATION — Confusion Matrices (Task A & B) + ROC/AUC (Task B)
# Drop this cell after the imports / data-loading section.
# Requires in scope: X_preprocessed, X_raw_scaled, X_cnn, pseudo_labels_dict,
#   y_task_B, build_inception_cnn, RandomForestClassifier, XGBClassifier,
#   LGBMClassifier, Input, Dense, Model, tf, np, torch, nn,
#   MAMBA_AVAILABLE  (and Mamba from mamba_ssm when True)
# ==============================================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, f1_score,
                             roc_curve, roc_auc_score)
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# ── Tunable knobs ──────────────────────────────────────────────────────────────
N_FOLDS         = 5
CV_EPOCHS_CNN   = 300   # 1500 in training; reduced for CV speed
CV_EPOCHS_AE    = 50
CV_EPOCHS_MAMBA = 200   # 500 in training; reduced for CV speed

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# ── Shared Mamba architecture (defined once, reused for both tasks) ────────────
if MAMBA_AVAILABLE:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    class MambaClassifier(nn.Module):
        def __init__(self, input_dim, num_cls):
            super().__init__()
            self.proj  = nn.Linear(input_dim, 64)
            self.mamba = Mamba(d_model=64, d_state=16, d_conv=4, expand=2)
            self.clf   = nn.Linear(64, num_cls)
        def forward(self, x):
            return self.clf(self.mamba(self.proj(x).unsqueeze(1)).squeeze(1))

    def train_mamba(X_tr, y_tr, input_dim, num_cls, epochs):
        """Train a MambaClassifier for `epochs` and return the fitted model."""
        model = MambaClassifier(input_dim, num_cls).to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
        crit  = nn.CrossEntropyLoss()
        Xt = torch.tensor(X_tr, dtype=torch.float32).to(device)
        yt = torch.tensor(y_tr, dtype=torch.long).to(device)
        for _ in range(epochs):
            model.train()
            opt.zero_grad()
            crit(model(Xt), yt).backward()
            opt.step()
        model.eval()
        return model

    def predict_mamba(model, X):
        """Return (class_preds, class1_proba) as numpy arrays."""
        with torch.no_grad():
            logits = model(torch.tensor(X, dtype=torch.float32).to(device))
            proba  = torch.softmax(logits, dim=1).cpu().numpy()
        return proba.argmax(axis=1), proba


# ── Model lists (Mamba appended only when the package is available) ─────────────
TASKA_MODELS = ['Prep_RF', 'Prep_XGB', 'Prep_LGBM', 'AE_XGBoost', 'Inception_CNN']
TASKB_MODELS = ['RF_FullSpectra', 'RF_GiniTop87', 'XGB_GiniTop87',
                'LGBM_GiniTop87', 'RF_AELatent',  'Inception_CNN']
if MAMBA_AVAILABLE:
    TASKA_MODELS = TASKA_MODELS + ['Mamba_SSM']
    TASKB_MODELS = TASKB_MODELS + ['Mamba_SSM']

# ==============================================================================
# TASK A — Nutrient Classification (N, P, K)  [3 classes: Low / Med / High]
# ==============================================================================
print("=" * 65)
print("TASK A: 5-FOLD OOF CROSS-VALIDATION  (N, P, K)")
print("=" * 65)

# OOF prediction arrays  (-1 = not yet filled)
taska_oof = {
    nut: {m: np.full(len(pseudo_labels_dict[nut]), -1, dtype=int)
          for m in TASKA_MODELS}
    for nut in ['N', 'P', 'K']
}

for nutrient in ['N', 'P', 'K']:
    y = pseudo_labels_dict[nutrient]
    print(f"\n  Nutrient {nutrient}  — fold:", end=" ", flush=True)

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_preprocessed, y)):
        print(fold_idx + 1, end=" ", flush=True)

        # ── Feature slices ────────────────────────────────────────────────────
        X_tr_pp,  X_val_pp  = X_preprocessed[tr_idx], X_preprocessed[val_idx]
        X_tr_raw, X_val_raw = X_raw_scaled[tr_idx],   X_raw_scaled[val_idx]
        X_tr_cnn, X_val_cnn = X_cnn[tr_idx],          X_cnn[val_idx]
        y_tr,     y_val     = y[tr_idx],               y[val_idx]

        # Per-fold Gini top-87 (fit on fold train only)
        rf_sel = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf_sel.fit(X_tr_pp, y_tr)
        top87 = np.argsort(rf_sel.feature_importances_)[::-1][:87]
        X_tr_ph,  X_val_ph  = X_tr_pp[:, top87], X_val_pp[:, top87]

        # Per-fold autoencoder latent (fit on fold train only)
        n_feat = X_raw_scaled.shape[1]
        inp_f  = Input(shape=(n_feat,))
        lat_f  = Dense(64,    activation='relu')(Dense(256, activation='relu')(inp_f))
        out_f  = Dense(n_feat, activation='linear')(Dense(256, activation='relu')(lat_f))
        ae_f   = Model(inp_f, out_f);  enc_f = Model(inp_f, lat_f)
        ae_f.compile(optimizer='adam', loss='mse')
        ae_f.fit(X_tr_raw, X_tr_raw, epochs=CV_EPOCHS_AE, batch_size=32, verbose=0)
        X_tr_lat  = enc_f.predict(X_tr_raw,  verbose=0)
        X_val_lat = enc_f.predict(X_val_raw, verbose=0)

        # ── Tree / boosting classifiers ───────────────────────────────────────
        fold_clfs_a = {
            'Prep_RF':    (RandomForestClassifier(n_estimators=100, max_depth=8,
                                                   min_samples_split=5,
                                                   random_state=42, n_jobs=-1),
                           X_tr_ph,  X_val_ph),
            'Prep_XGB':   (XGBClassifier(max_depth=6, learning_rate=0.1,
                                          random_state=42),
                           X_tr_ph,  X_val_ph),
            'Prep_LGBM':  (LGBMClassifier(max_depth=6, learning_rate=0.1,
                                           random_state=42, verbose=-1),
                           X_tr_ph,  X_val_ph),
            'AE_XGBoost': (XGBClassifier(max_depth=6, learning_rate=0.1,
                                          random_state=42),
                           X_tr_lat, X_val_lat),
        }
        for name, (clf, Xtr, Xval) in fold_clfs_a.items():
            clf.fit(Xtr, y_tr)
            taska_oof[nutrient][name][val_idx] = clf.predict(Xval)

        # ── Inception CNN ─────────────────────────────────────────────────────
        cnn_f = build_inception_cnn(input_shape=(X_cnn.shape[1], 1), num_classes=3)
        cnn_f.fit(X_tr_cnn, y_tr, epochs=CV_EPOCHS_CNN, batch_size=32, verbose=0)
        taska_oof[nutrient]['Inception_CNN'][val_idx] = np.argmax(
            cnn_f.predict(X_val_cnn, verbose=0), axis=1)
        tf.keras.backend.clear_session()

        # ── Mamba SSM (3-class, input = L2-normalised raw spectra) ────────────
        if MAMBA_AVAILABLE:
            X_tr_mb  = X_tr_cnn.squeeze(-1)   # (n_train, n_features)
            X_val_mb = X_val_cnn.squeeze(-1)
            mamba_a  = train_mamba(X_tr_mb, y_tr,
                                   input_dim=X_cnn.shape[1], num_cls=3,
                                   epochs=CV_EPOCHS_MAMBA)
            preds_mb, _ = predict_mamba(mamba_a, X_val_mb)
            taska_oof[nutrient]['Mamba_SSM'][val_idx] = preds_mb
            del mamba_a

    print()  # newline after fold numbers

# ── OOF summary table (Task A) ────────────────────────────────────────────────
print("\n  OOF performance summary:")
header = f"  {'Model':<16}"
for nut in ['N', 'P', 'K']:
    header += f"  {nut+' Acc':>7} {nut+' F1':>6}"
print(header)
print("  " + "-" * 70)
for m in TASKA_MODELS:
    row = f"  {m:<16}"
    for nut in ['N', 'P', 'K']:
        yt = pseudo_labels_dict[nut];  yp = taska_oof[nut][m]
        row += f"  {accuracy_score(yt,yp)*100:6.1f}%  {f1_score(yt,yp,average='macro')*100:5.1f}%"
    print(row)

# ── Confusion-matrix grid (Task A) ────────────────────────────────────────────
CLASS_LABELS_A = ['Low', 'Med', 'High']
n_rows_a, n_cols_a = len(['N', 'P', 'K']), len(TASKA_MODELS)

fig_a, axes_a = plt.subplots(n_rows_a, n_cols_a,
                              figsize=(4.8 * n_cols_a, 4.2 * n_rows_a))

for r, nutrient in enumerate(['N', 'P', 'K']):
    y_true = pseudo_labels_dict[nutrient]
    for c, model_name in enumerate(TASKA_MODELS):
        y_oof = taska_oof[nutrient][model_name]
        acc   = accuracy_score(y_true, y_oof)
        f1    = f1_score(y_true, y_oof, average='macro')
        cm    = confusion_matrix(y_true, y_oof)
        ax    = axes_a[r, c]
        ConfusionMatrixDisplay(cm, display_labels=CLASS_LABELS_A).plot(
            ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(f"{nutrient}  —  {model_name}\n"
                     f"Acc {acc*100:.1f}%  F1 {f1*100:.1f}%",
                     fontsize=9, pad=4)
        ax.set_xlabel("Predicted", fontsize=8)
        ax.set_ylabel("True",      fontsize=8)
        ax.tick_params(labelsize=8)

fig_a.suptitle("Task A (Internal) — 5-Fold OOF Confusion Matrices",
               fontsize=14, y=1.005, fontweight='bold')
plt.tight_layout()
plt.savefig("taska_oof_confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved → taska_oof_confusion_matrices.png")


# ==============================================================================
# TASK B — Contamination Detection  [binary: 0=Clean, 1=Contaminated]
# ==============================================================================
print("\n" + "=" * 65)
print("TASK B: 5-FOLD OOF CROSS-VALIDATION  (Clean vs Contaminated)")
print("=" * 65)

taskb_oof      = {m: np.full(len(y_task_B), -1, dtype=int)   for m in TASKB_MODELS}
taskb_oof_prob = {m: np.full(len(y_task_B),  np.nan)         for m in TASKB_MODELS}

print("\n  Fold:", end=" ", flush=True)
for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_preprocessed, y_task_B)):
    print(fold_idx + 1, end=" ", flush=True)

    X_tr_pp,  X_val_pp  = X_preprocessed[tr_idx], X_preprocessed[val_idx]
    X_tr_raw, X_val_raw = X_raw_scaled[tr_idx],   X_raw_scaled[val_idx]
    X_tr_cnn, X_val_cnn = X_cnn[tr_idx],          X_cnn[val_idx]
    y_tr,     y_val     = y_task_B[tr_idx],        y_task_B[val_idx]

    # Per-fold Gini top-87
    rf_sel_b = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_sel_b.fit(X_tr_pp, y_tr)
    top87_b  = np.argsort(rf_sel_b.feature_importances_)[::-1][:87]
    X_tr_ph,  X_val_ph  = X_tr_pp[:, top87_b], X_val_pp[:, top87_b]

    # Per-fold AE latent
    n_feat = X_raw_scaled.shape[1]
    inp_b  = Input(shape=(n_feat,))
    lat_b  = Dense(64,    activation='relu')(Dense(256, activation='relu')(inp_b))
    out_b  = Dense(n_feat, activation='linear')(Dense(256, activation='relu')(lat_b))
    ae_b   = Model(inp_b, out_b);  enc_b = Model(inp_b, lat_b)
    ae_b.compile(optimizer='adam', loss='mse')
    ae_b.fit(X_tr_raw, X_tr_raw, epochs=CV_EPOCHS_AE, batch_size=32, verbose=0)
    X_tr_lat  = enc_b.predict(X_tr_raw,  verbose=0)
    X_val_lat = enc_b.predict(X_val_raw, verbose=0)

    # ── Tree / boosting classifiers (store both class pred & class-1 proba) ───
    fold_clfs_b = {
        'RF_FullSpectra': (RandomForestClassifier(n_estimators=100, max_depth=8,
                                                   random_state=42, n_jobs=-1),
                           X_tr_pp,  X_val_pp),
        'RF_GiniTop87':   (RandomForestClassifier(n_estimators=100, max_depth=8,
                                                   random_state=42, n_jobs=-1),
                           X_tr_ph,  X_val_ph),
        'XGB_GiniTop87':  (XGBClassifier(max_depth=6, learning_rate=0.1,
                                          random_state=42),
                           X_tr_ph,  X_val_ph),
        'LGBM_GiniTop87': (LGBMClassifier(max_depth=6, learning_rate=0.1,
                                           random_state=42, verbose=-1),
                           X_tr_ph,  X_val_ph),
        'RF_AELatent':    (RandomForestClassifier(n_estimators=100, max_depth=8,
                                                   random_state=42, n_jobs=-1),
                           X_tr_lat, X_val_lat),
    }
    for name, (clf, Xtr, Xval) in fold_clfs_b.items():
        clf.fit(Xtr, y_tr)
        taskb_oof[name][val_idx]      = clf.predict(Xval)
        taskb_oof_prob[name][val_idx] = clf.predict_proba(Xval)[:, 1]

    # ── Inception CNN (binary) ────────────────────────────────────────────────
    cnn_b_f = build_inception_cnn(input_shape=(X_cnn.shape[1], 1), num_classes=2)
    cnn_b_f.fit(X_tr_cnn, y_tr, epochs=CV_EPOCHS_CNN, batch_size=32, verbose=0)
    prob_cnn = cnn_b_f.predict(X_val_cnn, verbose=0)          # (n_val, 2)
    taskb_oof['Inception_CNN'][val_idx]      = prob_cnn.argmax(axis=1)
    taskb_oof_prob['Inception_CNN'][val_idx] = prob_cnn[:, 1]
    tf.keras.backend.clear_session()

    # ── Mamba SSM (2-class, input = L2-normalised raw spectra) ───────────────
    if MAMBA_AVAILABLE:
        X_tr_mb  = X_tr_cnn.squeeze(-1)
        X_val_mb = X_val_cnn.squeeze(-1)
        mamba_b  = train_mamba(X_tr_mb, y_tr,
                               input_dim=X_cnn.shape[1], num_cls=2,
                               epochs=CV_EPOCHS_MAMBA)
        preds_mb, prob_mb = predict_mamba(mamba_b, X_val_mb)
        taskb_oof['Mamba_SSM'][val_idx]      = preds_mb
        taskb_oof_prob['Mamba_SSM'][val_idx] = prob_mb[:, 1]
        del mamba_b

print()

# ── OOF summary table (Task B) ────────────────────────────────────────────────
print("\n  OOF performance summary:")
print(f"  {'Model':<18} {'Acc':>8} {'Macro-F1':>10} {'ROC-AUC':>10}")
print("  " + "-" * 52)
for m in TASKB_MODELS:
    acc  = accuracy_score(y_task_B, taskb_oof[m])
    f1   = f1_score(y_task_B, taskb_oof[m], average='macro')
    auc  = roc_auc_score(y_task_B, taskb_oof_prob[m])
    print(f"  {m:<18}  {acc*100:6.1f}%   {f1*100:6.1f}%   {auc:.4f}")

# ── Confusion-matrix grid (Task B) ────────────────────────────────────────────
CLASS_LABELS_B = ['Clean', 'Contam.']
N_COLS_B = 3 if not MAMBA_AVAILABLE else 4   # 6 or 7 models → tighter grid
N_ROWS_B = (len(TASKB_MODELS) + N_COLS_B - 1) // N_COLS_B

fig_b, axes_b = plt.subplots(N_ROWS_B, N_COLS_B,
                              figsize=(4.8 * N_COLS_B, 4.2 * N_ROWS_B))
axes_b_flat = axes_b.flatten() if N_ROWS_B > 1 else np.array(axes_b).flatten()

for i, model_name in enumerate(TASKB_MODELS):
    y_oof = taskb_oof[model_name]
    acc   = accuracy_score(y_task_B, y_oof)
    f1    = f1_score(y_task_B,       y_oof, average='macro')
    auc   = roc_auc_score(y_task_B,  taskb_oof_prob[model_name])
    cm    = confusion_matrix(y_task_B, y_oof)
    ax    = axes_b_flat[i]
    ConfusionMatrixDisplay(cm, display_labels=CLASS_LABELS_B).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{model_name}\n"
                 f"Acc {acc*100:.1f}%  F1 {f1*100:.1f}%  AUC {auc:.3f}",
                 fontsize=9, pad=4)
    ax.set_xlabel("Predicted", fontsize=8)
    ax.set_ylabel("True",      fontsize=8)
    ax.tick_params(labelsize=8)

for j in range(len(TASKB_MODELS), len(axes_b_flat)):
    axes_b_flat[j].set_visible(False)

fig_b.suptitle("Task B (Contamination) — 5-Fold OOF Confusion Matrices",
               fontsize=14, y=1.005, fontweight='bold')
plt.tight_layout()
plt.savefig("taskb_oof_confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved → taskb_oof_confusion_matrices.png")

# ── ROC / AUC figure (Task B) ─────────────────────────────────────────────────
# Colour palette — enough distinct colours for up to 7 models
PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
           '#9467bd', '#8c564b', '#e377c2']

fig_roc, ax_roc = plt.subplots(figsize=(7, 6))

for idx, model_name in enumerate(TASKB_MODELS):
    fpr, tpr, _ = roc_curve(y_task_B, taskb_oof_prob[model_name])
    auc_val      = roc_auc_score(y_task_B, taskb_oof_prob[model_name])
    ax_roc.plot(fpr, tpr,
                color=PALETTE[idx % len(PALETTE)],
                lw=2,
                label=f"{model_name}  (AUC = {auc_val:.3f})")

# Diagonal reference line
ax_roc.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.500)')

ax_roc.set_xlim([0.0, 1.0])
ax_roc.set_ylim([0.0, 1.05])
ax_roc.set_xlabel("False Positive Rate", fontsize=12)
ax_roc.set_ylabel("True Positive Rate",  fontsize=12)
ax_roc.set_title("Task B — ROC Curves (5-Fold OOF)\nClean vs Contaminated",
                 fontsize=13, fontweight='bold')
ax_roc.legend(loc='lower right', fontsize=9, framealpha=0.85)
ax_roc.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("taskb_oof_roc_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → taskb_oof_roc_curves.png")

In [ ]:
import os
import shutil
from IPython.display import FileLink, display

# 1. Define the exact target directories
directories = [
    "/kaggle/working/FYP_Exported_Models_TaskA",
    "/kaggle/working/FYP_Exported_Models_TaskB",
    "/kaggle/working/FYP_Exported_Models_TaskA_Ext"
]

print("\n--- ZIPPING OUTPUTS ---")
zip_paths = []

for folder_path in directories:
    # Strip any trailing slashes to prevent naming issues
    clean_path = folder_path.rstrip('/')
    
    if os.path.exists(clean_path):
        # shutil.make_archive(output_filename, 'format', source_dir)
        # This creates /kaggle/working/FolderName.zip
        archive_path = shutil.make_archive(clean_path, 'zip', clean_path)
        zip_paths.append(archive_path)
        print(f"Successfully created: {archive_path}")
    else:
        print(f"Warning: Directory not found -> {clean_path}")

print("\n--- DOWNLOAD LINKS ---")
# 2. Generate clickable links for the newly created zip files
for zip_file in zip_paths:
    # FileLink requires the path relative to the notebook server root (/kaggle/working/)
    relative_path = os.path.relpath(zip_file, start='/kaggle/working')
    display(FileLink(relative_path))

print("\nTRAINING & EXPORT PIPELINE COMPLETE.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches

# ==============================================================================
# 1. GENERATE RECONSTRUCTION
# ==============================================================================
# Select a representative sample (e.g., the first sample, or you can use the mean)
sample_idx = 5  # Feel free to change this index to view different samples
original_spectrum = X_raw_scaled[sample_idx]  # Use X_prep_ext if doing Task A Ext

# Predict (reconstruct) the spectrum using the trained autoencoder
reconstructed_spectrum = autoencoder.predict(original_spectrum.reshape(1, -1), verbose=0)[0]

# ==============================================================================
# 2. DEFINE NIST PEAKS FOR HIGHLIGHTING
# ==============================================================================
nist_lines = {
    'Nitrogen (N)': {'color': 'blue', 'peaks': [746.83, 821.63, 868.34]},
    'Phosphorus (P)': {'color': 'green', 'peaks': [213.62, 253.56, 255.33]},
    'Potassium (K)': {'color': 'red', 'peaks': [766.49, 769.90]}
}
tolerance = 2.0  # Width of the shaded region in nm

# ==============================================================================
# 3. CREATE THE PLOT
# ==============================================================================
plt.figure(figsize=(14, 6))

# Plot Original vs Reconstructed
plt.plot(wavelengths, original_spectrum, label='Original Input Spectrum', color='black', linewidth=1.5, alpha=0.8)
plt.plot(wavelengths, reconstructed_spectrum, label='Autoencoder Reconstruction', color='darkorange', linewidth=1.5, linestyle='--')

# Add NIST highlighting
for element, config in nist_lines.items():
    added_label = False
    for peak in config['peaks']:
        # Only add the label once per element to avoid legend clutter
        label = f"{element} Peaks" if not added_label else None
        plt.axvspan(peak - tolerance, peak + tolerance, color=config['color'], alpha=0.2, label=label)
        added_label = True

# Formatting
plt.title('Autoencoder Reconstruction Fidelity vs. Original LIBS Spectrum', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Wavelength (nm)', fontsize=12, fontweight='bold')
plt.ylabel('Normalized Intensity', fontsize=12, fontweight='bold')
plt.xlim(wavelengths.min(), wavelengths.max())

# Optimize Legend
plt.legend(loc='upper right', fontsize=11, framealpha=0.9, edgecolor='black')
plt.grid(axis='both', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('autoencoder_reconstruction_proof.png', dpi=300)
plt.show()

# ==============================================================================
# 4. CALCULATE RECONSTRUCTION ERROR (For Report Text)
# ==============================================================================
mse = np.mean(np.square(original_spectrum - reconstructed_spectrum))
print(f"Mean Squared Error (MSE) for Sample {sample_idx}: {mse:.6f}")

In [ ]:
loaded_model_B = joblib.load(os.path.join(TASKB_DIR, "LightGBM_GiniTop87_TaskB.joblib"))

# 2. Pass the model object to m2cgen
c_code_B = m2c.export_to_c(loaded_model_B)

# 3. Write it to a file
with open("Prep_LGBM_B.c", "w") as f:
    f.write(c_code_B)

In [ ]:
from IPython.display import FileLink
display(FileLink("Prep_LGBM_N.c"))
display(FileLink("Prep_LGBM_P.c"))
display(FileLink("Prep_LGBM_K.c"))
display(FileLink("Prep_LGBM_B.c"))

**Testing**

In [ ]:
import os
import warnings
import tensorflow as tf
import numpy as np
import pandas as pd
import joblib
import subprocess
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.stats import kruskal
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

import torch
import torch.nn as nn
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
except ImportError:
    MAMBA_AVAILABLE = False

try:
    import m2cgen as m2c
except ImportError:
    m2c = None

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# ==============================================================================
# 0. DIRECTORY SETUP & DATA LOADING
# ==============================================================================
TASKA_DIR = "/kaggle/input/datasets/ryancclh/fyp-exported-models-taska/"
TASKB_DIR = "/kaggle/input/datasets/ryancclh/fyp-exported-models-taskb/"
TASKA_EXT_DIR = "/kaggle/input/datasets/ryancclh/fyp-exported-models-taska-ext/"

print(f"Loading Internal Irish Soil Data...")
df = pd.read_csv("/kaggle/input/datasets/ryancclh/irish-soil-dataset/Irish_Soil_DataSet2.csv", index_col=0)
X_raw = df.T.values.astype(float)
wavelengths = df.T.columns.values.astype(float)
sample_names = df.T.index.values

print("Loading Task A Cached Internal Models...")
X_preprocessed = np.load(os.path.join(TASKA_DIR, 'X_preprocessed_full.npy'))
X_raw_scaled = np.load(os.path.join(TASKA_DIR, 'X_raw_scaled_full.npy'))
X_cnn = normalize(X_raw, norm='l2', axis=1).reshape((X_raw.shape[0], X_raw.shape[1], 1))

pseudo_labels_dict = joblib.load(os.path.join(TASKA_DIR, 'pseudo_labels_dict.joblib'))
encoder = tf.keras.models.load_model(os.path.join(TASKA_DIR, 'global_encoder.keras'))
X_latent = encoder.predict(X_raw_scaled, verbose=0)

y_task_B = np.array([1 if any(c in name for c in ["ChalkT", "Arsenic", "TPH", "GardenTop_w_HeavyMetals"]) else 0 for name in sample_names])

# Load Internal A Models
loaded_A = {'N': {}, 'P': {}, 'K': {}}
indices_A = {}
for nut in ['N', 'P', 'K']:
    indices_A[nut] = np.load(os.path.join(TASKA_DIR, f'top87_indices_{nut}.npy'))
    for clf in ['Prep_RF', 'Prep_XGB', 'Prep_LGBM', 'AE_XGBoost']:
        loaded_A[nut][clf] = joblib.load(os.path.join(TASKA_DIR, f"{clf}_{nut}.joblib"))
    loaded_A[nut]['Inception_CNN'] = tf.keras.models.load_model(os.path.join(TASKA_DIR, f"Inception_CNN_{nut}.keras"))

# Load Task B Models
top87_b = np.load(os.path.join(TASKB_DIR, "top87_indices_TaskB.npy"))
rf_b = joblib.load(os.path.join(TASKB_DIR, "RandomForest_GiniTop87_TaskB.joblib"))


# ==============================================================================
# 1. TASK A: INTERNAL VALIDITY & METRICS (TEST SET EVALUATION)
# ==============================================================================
print("\n" + "="*60)
print("PHASE 1: TASK A (N, P, K) TEST SET EVALUATION")
print("="*60)

for nutrient in ['N', 'P', 'K']:
    print(f"\n--- {nutrient} CLASSIFIER ---")
    y = pseudo_labels_dict[nutrient]
    
    # Accurate train/test splits derived strictly for metrics
    _, X_te, _, y_te = train_test_split(X_preprocessed, y, test_size=0.2, random_state=42, stratify=y)
    _, X_te_cnn, _, _ = train_test_split(X_cnn, y, test_size=0.2, random_state=42, stratify=y)
    _, X_te_lat, _, _ = train_test_split(X_latent, y, test_size=0.2, random_state=42, stratify=y)
    
    preds_dict = {
        'Prep_RF': loaded_A[nutrient]['Prep_RF'].predict(X_te[:, indices_A[nutrient]]),
        'Prep_XGB': loaded_A[nutrient]['Prep_XGB'].predict(X_te[:, indices_A[nutrient]]),
        'Prep_LGBM': loaded_A[nutrient]['Prep_LGBM'].predict(X_te[:, indices_A[nutrient]]),
        'AE_XGBoost': loaded_A[nutrient]['AE_XGBoost'].predict(X_te_lat),
        'Inception_CNN': np.argmax(loaded_A[nutrient]['Inception_CNN'].predict(X_te_cnn, verbose=0), axis=1)
    }
    
    for name, preds in preds_dict.items():
        print(f" {name:15s} | Acc: {accuracy_score(y_te, preds)*100:.2f}% | Macro F1: {f1_score(y_te, preds, average='macro')*100:.2f}%")


# ==============================================================================
# 2. TASK A: EXTERNAL DATASET VALIDATION (P and K)
# ==============================================================================
print("\n" + "="*60)
print("PHASE 2: EXTERNAL CONCENTRATION VALIDATION")
print("="*60)

LIBS_DATA_PATH = "/kaggle/input/datasets/ryancclh/libs-extra-dataset/LIBS Extra Dataset/LIBS data.xlsx"
FERTILITY_DATA_PATH = "/kaggle/input/datasets/ryancclh/libs-extra-dataset/LIBS Extra Dataset/soil fertility data.xlsx"

df_libs = pd.read_excel(LIBS_DATA_PATH, header=None)
wavelengths_ext = df_libs.iloc[2:, 0].values.astype(float)
X_raw_ext = df_libs.iloc[2:, 1:].values.astype(float).T
X_prep_ext = normalize(np.maximum(savgol_filter(X_raw_ext, 15, 3, axis=1), 0), norm='l2', axis=1)
X_cnn_ext = X_prep_ext.reshape((X_prep_ext.shape[0], X_prep_ext.shape[1], 1))

df_fert = pd.read_excel(FERTILITY_DATA_PATH)
true_ext = {'P': df_fert['exP'].values.astype(float), 'K': df_fert['exK'].values.astype(float)}

pseudo_ext = joblib.load(os.path.join(TASKA_EXT_DIR, 'pseudo_labels_ext.joblib'))
encoder_ext = tf.keras.models.load_model(os.path.join(TASKA_EXT_DIR, 'global_encoder_ext.keras'))
X_lat_ext = encoder_ext.predict(X_prep_ext, verbose=0)

if MAMBA_AVAILABLE:
    class MambaClassifier(nn.Module):
        def __init__(self, input_dim, num_cls=3):
            super().__init__()
            self.proj = nn.Linear(input_dim, 64)
            self.mamba = Mamba(d_model=64, d_state=16, d_conv=4, expand=2)
            self.clf = nn.Linear(64, num_cls)
        def forward(self, x):
            return self.clf(self.mamba(self.proj(x).unsqueeze(1)).squeeze(1))

for element in ['P', 'K']:
    print(f"\n--- EVALUATING EXTERNAL PIPELINE FOR: {element} ---")
    y_ext = pseudo_ext[element]
    _, X_te, _, y_te, _, true_te = train_test_split(X_prep_ext, y_ext, true_ext[element], test_size=0.2, random_state=42, stratify=y_ext)
    _, X_te_cnn, _, _ = train_test_split(X_cnn_ext, y_ext, test_size=0.2, random_state=42, stratify=y_ext)
    _, X_te_lat, _, _ = train_test_split(X_lat_ext, y_ext, test_size=0.2, random_state=42, stratify=y_ext)
    
    top87_ext = np.load(os.path.join(TASKA_EXT_DIR, f'top87_indices_ext_{element}.npy'))
    X_te_phys = X_te[:, top87_ext]

    rf_ext = joblib.load(os.path.join(TASKA_EXT_DIR, f"Prep_RF_ext_{element}.joblib"))
    xgb_ext = joblib.load(os.path.join(TASKA_EXT_DIR, f"Prep_XGB_ext_{element}.joblib"))
    lgbm_ext = joblib.load(os.path.join(TASKA_EXT_DIR, f"Prep_LGBM_ext_{element}.joblib"))
    ae_xgb_ext = joblib.load(os.path.join(TASKA_EXT_DIR, f"AE_XGBoost_ext_{element}.joblib"))
    cnn_ext = tf.keras.models.load_model(os.path.join(TASKA_EXT_DIR, f"Inception_CNN_ext_{element}.keras"))
    
    preds_dict = {
        "Direct Peak Inspection": y_te,  
        "Prep + RF Gini 87": rf_ext.predict(X_te_phys),
        "Prep + XGB Gini 87": xgb_ext.predict(X_te_phys),
        "Prep + LGBM Gini 87": lgbm_ext.predict(X_te_phys),
        "AE + XGB": ae_xgb_ext.predict(X_te_lat),
        "1D CNN (TensorFlow)": np.argmax(cnn_ext.predict(X_te_cnn, verbose=0), axis=1)
    }

    if MAMBA_AVAILABLE:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        m_model_ext = MambaClassifier(X_prep_ext.shape[1], 3).to(device)
        m_model_ext.load_state_dict(torch.load(os.path.join(TASKA_EXT_DIR, f"MambaSSM_ext_{element}.pth"), map_location=device))
        m_model_ext.eval()
        with torch.no_grad():
            preds_dict["Mamba SSM"] = torch.argmax(m_model_ext(torch.tensor(X_te, dtype=torch.float32).to(device)), dim=1).cpu().numpy()

    final_metrics = []
    for model_name, preds in preds_dict.items():
        df_eval = pd.DataFrame({'Predicted': preds, 'True_Val': true_te})
        df_eval['Label_Text'] = df_eval['Predicted'].map({0: "Low", 1: "Medium", 2: "High"})
        means = df_eval.groupby('Label_Text')['True_Val'].mean()
        m_low, m_med, m_high = means.get('Low', float('nan')), means.get('Medium', float('nan')), means.get('High', float('nan'))
        
        try:
            _, p_val = kruskal(df_eval[df_eval['Label_Text'] == 'Low']['True_Val'].dropna(), df_eval[df_eval['Label_Text'] == 'High']['True_Val'].dropna())
        except:
            p_val = 1.0
            
        final_metrics.append({
            "Model": model_name,
            "Low (mg/kg)": round(m_low, 2), "Med (mg/kg)": round(m_med, 2), "High (mg/kg)": round(m_high, 2),
            "Monotonic?": "✅ Yes" if (m_low < m_med < m_high) else "❌ No",
            "Stat Sig": "✅ Yes" if p_val < 0.05 else "❌ No"
        })
    print(pd.DataFrame(final_metrics).to_string(index=False))

# ==============================================================================
# 3. TASK B: CONTAMINANT SET EVALUATION
# ==============================================================================
print("\n" + "="*60)
print("TASK B: CONTAMINATION TEST SET EVALUATION")
print("="*60)

_, X_te_b, _, y_te_b = train_test_split(X_preprocessed, y_task_B, test_size=0.2, random_state=42, stratify=y_task_B)
acc = accuracy_score(y_te_b, rf_b.predict(X_te_b[:, top87_b]))
print(f"Task B Random Forest | Acc: {acc*100:.2f}% (Target Check: > 73.01%)")

print("\nEvaluation pipeline complete.")

In [ ]:
"""
radar_efficiency.py
--------------------
Side-by-side radar charts comparing 5 analytical-profiling metrics
across ML models for Task A (NPK classification) and Task B
(contamination detection).

Axes (all 0-100, 100 = best):
  Accuracy  = (balanced acc% - 70) / 30 * 100  [70% -> 0, 100% -> 100]
  Params    = log-normalised inverse of parameter count   (fewer -> higher)
  MACs      = log-normalised inverse of MACs inference    (fewer -> higher)
  RAM       = log-normalised inverse of peak RAM (kB)     (less  -> higher)
  Flash     = log-normalised inverse of flash size (kB)   (less  -> higher)

Sources:
  Accuracy  -> Table 4.3 (Task A, balanced accuracy avg N/P/K)
             -> Table 4.5 (Task B, best variant per architecture)
  Hardware  -> Table 4.6 (Task A), Table 4.7 (Task B)

  Filled marker  = model present in profiling tables (may have N/A metrics)
  Hollow marker  = model absent from profiling tables; hardware scores estimated
  N/A values in tables (LGBM params/MACs; CNN MACs) also estimated from
  model family characteristics.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
AXES   = ['Accuracy', 'Params', 'MACs', 'RAM', 'Flash']
N_AXES = len(AXES)

TASKS = {
    'A': {
        'title':    'Task A — NPK classification',
        'subtitle': 'Balanced acc avg N, P, K  ·  Tables 4.3 & 4.6',
        'models': [
            # (name,               scores,                    in_profiling_table)
            ('Prep + RF',          [44, 100, 100, 100, 100],  True ),
            ('Prep + XGBoost',     [52,  90,  90, 100,  90],  False),  # absent Table 4.6
            ('Prep + LightGBM',    [52,  85,  85, 100,  82],  True ),
            ('Inception CNN',      [56,   0,  20,   0,   0],  True ),
            ('AE + XGBoost',       [48,   6,  30,   4,  13],  True ),
        ],
    },
    'B': {
        'title':    'Task B — contamination detection',
        'subtitle': 'Best variant accuracy  ·  Tables 4.5 & 4.7',
        'models': [
            ('Random Forest',  [83, 100, 100, 100,  91],  True ),
            ('LightGBM',       [89,  85,  85, 100, 100],  True ),
            ('XGBoost',        [90,  80,  85, 100,  85],  False),  # absent Table 4.7
            ('Inception CNN',  [96,   0,  20,   0,   0],  True ),
            ('Mamba',          [78,  70,  65,  90,  75],  False),  # absent Table 4.7
        ],
    },
}

# ---------------------------------------------------------------------------
# Per-model visual encoding
#   Color       → model family (consistent across both charts)
#   Line style  → separates overlapping polygons
#   Marker      → unique vertex shape so overlapping lines stay identifiable
#   Filled/open → whether the model is in the profiling tables
# ---------------------------------------------------------------------------
COLORS      = ['#378ADD', '#639922', '#BA7517', '#D85A30', '#534AB7']
LINE_STYLES = [
    '-',              # 0  solid
    '--',             # 1  dashed
    '-.',             # 2  dash-dot
    (0, (5, 1.5)),    # 3  long-dash
    (0, (1, 1.5)),    # 4  dense dotted
]
LINE_WIDTHS  = [2.4, 2.1, 2.1, 2.3, 1.9]
MARKERS      = ['o', 's', '^', 'D', 'v']   # circle, square, tri-up, diamond, tri-down
MARKER_SIZES = [42,  38,  46,  36,  42]    # area in pts²
FILL_ALPHAS  = [0.14, 0.09, 0.09, 0.12, 0.08]

# Draw order for fills: larger/fuller polygons first, CNN (spiky) on top
DRAW_ORDER = [0, 1, 2, 4, 3]

# ---------------------------------------------------------------------------
# Angle array
# ---------------------------------------------------------------------------
angles        = np.linspace(0, 2 * np.pi, N_AXES, endpoint=False).tolist()
angles_closed = angles + angles[:1]

# ---------------------------------------------------------------------------
# draw_radar
# ---------------------------------------------------------------------------
def draw_radar(ax, task_dict):
    ax.set_facecolor('white')
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 100)
    ax.set_yticks([25, 50, 75, 100])
    ax.set_yticklabels(['25', '50', '75', '100'], fontsize=7.5, color='#888780')
    ax.set_xticks(angles)
    ax.set_xticklabels(AXES, fontsize=11, fontweight='500', color='#2C2C2A')
    ax.tick_params(axis='y', labelsize=7)
    ax.grid(color='#D3D1C7', linewidth=0.55, linestyle='-', alpha=0.75)
    ax.spines['polar'].set_color('#D3D1C7')
    ax.set_rlabel_position(22)

    models = task_dict['models']

    # Fills — drawn largest first so distinctive shapes remain visible
    for i in DRAW_ORDER:
        _, scores, _ = models[i]
        ax.fill(angles_closed, scores + scores[:1],
                color=COLORS[i], alpha=FILL_ALPHAS[i], zorder=2)

    # Lines
    for i in DRAW_ORDER:
        _, scores, _ = models[i]
        ax.plot(angles_closed, scores + scores[:1],
                color=COLORS[i],
                linewidth=LINE_WIDTHS[i],
                linestyle=LINE_STYLES[i],
                zorder=3 + i * 0.1)

    # Markers + legend handles
    legend_handles = []
    for i, (name, scores, measured) in enumerate(models):
        mfc = COLORS[i] if measured else 'white'
        ax.scatter(angles, scores,
                   s=MARKER_SIZES[i],
                   marker=MARKERS[i],
                   facecolors=mfc,
                   edgecolors=COLORS[i],
                   linewidths=1.6,
                   zorder=5,
                   clip_on=False)

        label = name + ('' if measured else '  †')
        legend_handles.append(
            Line2D([0], [0],
                   color=COLORS[i],
                   linewidth=2,
                   linestyle=LINE_STYLES[i],
                   marker=MARKERS[i],
                   markersize=6,
                   markerfacecolor=mfc,
                   markeredgecolor=COLORS[i],
                   markeredgewidth=1.4,
                   label=label)
        )

    return legend_handles


# ---------------------------------------------------------------------------
# Figure — explicit subplots_adjust so title, legend, and footer never clash
# ---------------------------------------------------------------------------
fig = plt.figure(figsize=(14, 8.5))
fig.patch.set_facecolor('white')

# Reserve space: top 15% for titles, bottom 28% for legend + footer
plt.subplots_adjust(left=0.04, right=0.96, top=0.82, bottom=0.28, wspace=0.25)

for col, key in enumerate(TASKS):
    task = TASKS[key]
    ax   = fig.add_subplot(1, 2, col + 1, polar=True)

    handles = draw_radar(ax, task)

    # ── Titles via fig.text so they never collide with chart content ───────────
    # ax.get_position() returns the axes bounding box in figure coordinates
    pos = ax.get_position()
    cx  = (pos.x0 + pos.x1) / 2   # horizontal centre of this subplot

    fig.text(cx, 0.93,
             task['title'],
             ha='center', va='bottom',
             fontsize=12.5, fontweight='500', color='#2C2C2A')

    fig.text(cx, 0.88,
             task['subtitle'],
             ha='center', va='bottom',
             fontsize=9, color='#888780', style='italic')

    # ── Legend — placed well below the chart ──────────────────────────────────
    ax.legend(
        handles=handles,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.18),
        fontsize=9.5,
        frameon=False,
        ncol=3,                  # 2 rows for 5 items — more compact than ncol=2
        handlelength=2.4,
        handletextpad=0.55,
        labelcolor='#444441',
        borderpad=0,
        columnspacing=1.0,
    )

# ---------------------------------------------------------------------------
# Footer — sits in the 28% bottom margin, safely below all legends
# ---------------------------------------------------------------------------
NOTE = (
    'All axes 0–100 (100 = best).   '
    'Accuracy = (balanced acc% − 70) ÷ 30 × 100.   '
    'Hardware efficiency = log-normalised inverse (fewer resources → higher score).\n'
    '† Absent from profiling tables — hardware scores estimated from model family.   '
    'Hollow markers = estimated.   '
    'N/A values in tables (LGBM params/MACs; CNN MACs) also partially estimated.'
)
fig.text(0.5, 0.02,
         NOTE,
         ha='center', va='bottom',
         fontsize=8.5, color='#888780', linespacing=1.55)

# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------
plt.savefig('radar_efficiency_comparison.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved → radar_efficiency_comparison.png')